In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install pandas pyarrow lxml pycountry -q
# import fulltext_affiliations

In [ ]:
#!/usr/bin/env python3
"""
fulltext_affiliations.py  —  Google Colab
=========================================
Recover AFFILIATIONS from the full-text files the harvester saved, to enlarge
coverage for studies whose OpenAlex metadata had no/weak affiliation data.

Extraction ladder (best structure first), per study, using raw/<oa_id>.{xml,pdf}:
  1. JATS <aff> + <contrib><xref ref-type="aff">   -> author-ALIGNED, w/ country
  2. GROBID TEI <affiliation>/<country> (if raw is PDF and GROBID up) -> aligned
  3. front-matter regex + pycountry gazetteer on PDF/HTML text -> paper-level BAG

Design choices (per your selection):
  * EXTRACT FOR ALL studies; keep OpenAlex and full-text affiliations SIDE BY SIDE
    so you can measure added coverage + disagreement (a data-paper result).
  * Country + institution + author-alignment WHERE POSSIBLE (JATS/TEI); PDF text
    yields a paper-level country bag (not author-aligned) — flagged as such.
  * Every record carries PROVENANCE + CONFIDENCE so leadership metrics can use
    only the reliable (author-aligned) rows and the involvement layer can use all.

OUTPUT
  fulltext_affiliations.parquet   per study:
    oa_id, ft_source, aff_method (jats|tei|regex), author_aligned (bool),
    ft_first_author_countries, ft_last_author_countries, ft_all_countries,
    ft_institutions, oa_all_countries (from metadata for side-by-side),
    added_countries (in FT not in OA), agrees (overlap>0), confidence

SETUP
  !pip install pandas pyarrow lxml pycountry -q
  # GROBID optional (only needed to re-parse PDFs for aligned affs); regex works without.
"""

import json, re, sys
from pathlib import Path
import pandas as pd
import pycountry
from lxml import etree

# =========================================================================== #
# CONFIG                                                                        #
# =========================================================================== #
# IN_PARQUET  = "/content/drive/MyDrive/.../with_topics.parquet"   # for OA affs + author order
# FT_DIR      = "/content/drive/MyDrive/.../fulltext"
IN_PARQUET  = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/authorships_reharvested.csv"   # any parquet w/ ids
FT_DIR     = "/content/drive/MyDrive/COGNOSX/ticks/fulltext"
RAW_DIR     = FT_DIR + "/raw"
SEC_DIR     = FT_DIR + "/sections"
OUT_PARQUET = FT_DIR + "/fulltext_affiliations.parquet"
GROBID_URL  = ""      # set to re-run GROBID on PDFs for author-aligned affs; else regex


def S(x):
    return "" if x is None or (isinstance(x, float) and pd.isna(x)) or x is pd.NA else str(x)
def short_id(x): return S(x).rsplit("/", 1)[-1]
def _txt(el): return re.sub(r"\s+", " ", "".join(el.itertext())).strip() if el is not None else ""


# =========================================================================== #
# country resolution (gazetteer + demonym/adjective patch)                    #
# =========================================================================== #
_COUNTRY_NAMES = {}
for c in pycountry.countries:
    _COUNTRY_NAMES[c.name.lower()] = c.alpha_2
    if hasattr(c, "official_name"): _COUNTRY_NAMES[c.official_name.lower()] = c.alpha_2
    if hasattr(c, "common_name"): _COUNTRY_NAMES[c.common_name.lower()] = c.alpha_2
# common variants/adjectives seen in affiliation strings
_COUNTRY_PATCH = {
    "usa": "US", "u.s.a": "US", "u.s.a.": "US", "united states": "US", "u.s.": "US",
    "uk": "GB", "u.k.": "GB", "united kingdom": "GB", "england": "GB", "scotland": "GB",
    "wales": "GB", "northern ireland": "GB", "great britain": "GB",
    "south korea": "KR", "korea": "KR", "republic of korea": "KR",
    "russia": "RU", "iran": "IR", "syria": "SY", "tanzania": "TZ", "vietnam": "VN",
    "bolivia": "BO", "venezuela": "VE", "moldova": "MD", "laos": "LA", "brunei": "BN",
    "ivory coast": "CI", "cote d'ivoire": "CI", "côte d'ivoire": "CI",
    "democratic republic of congo": "CD", "drc": "CD", "dr congo": "CD",
    "republic of congo": "CG", "the gambia": "GM", "gambia": "GM", "cape verde": "CV",
    "cabo verde": "CV", "swaziland": "SZ", "eswatini": "SZ", "czech republic": "CZ",
    "czechia": "CZ", "the netherlands": "NL", "netherlands": "NL",
    "p.r. china": "CN", "pr china": "CN", "p.r.china": "CN", "prc": "CN",
    "people's republic of china": "CN", "mainland china": "CN",
}
# build a longest-first list for greedy matching
_LOOKUP = dict(_COUNTRY_NAMES); _LOOKUP.update(_COUNTRY_PATCH)
_SORTED_NAMES = sorted(_LOOKUP.keys(), key=len, reverse=True)

def countries_in(text):
    """Return set of ISO2 codes whose country name/variant appears in the text."""
    t = " " + re.sub(r"\s+", " ", S(text).lower()) + " "
    found = set()
    for name in _SORTED_NAMES:
        # allow short (3-char) names only if they are explicit patch variants
        # (usa, drc, prc, uk...) which are unambiguous; skip generic short tokens
        if len(name) < 4 and name not in _COUNTRY_PATCH:
            continue
        if len(name) < 2:
            continue
        if re.search(r"(?<![a-z])" + re.escape(name) + r"(?![a-z])", t):
            found.add(_LOOKUP[name])
    return found

def last_country_in_line(line):
    """Affiliation lines usually END with the country -> take the match at the
    LATEST position in the line (across all country names/variants)."""
    t = S(line).lower()
    best, best_pos = "", -1
    for name in _SORTED_NAMES:
        if len(name) < 4 and name not in _COUNTRY_PATCH:
            continue
        for m in re.finditer(r"(?<![a-z])" + re.escape(name) + r"(?![a-z])", t):
            if m.start() > best_pos:
                best, best_pos = _LOOKUP[name], m.start()
    return best


# =========================================================================== #
# 1. JATS: author-aligned affiliations (best)                                 #
# =========================================================================== #
def from_jats(xml_text):
    try:
        xml_text = re.sub(r"<!DOCTYPE[^>]*>", "", xml_text)
        root = etree.fromstring(xml_text.encode("utf-8"),
                                parser=etree.XMLParser(recover=True, resolve_entities=False))
    except Exception:
        return None
    # aff id -> (institution, country)
    affs = {}
    for aff in root.xpath(".//aff"):
        aid = aff.get("id") or ""
        inst = _txt(aff.find(".//institution")) or ""
        ctry_el = aff.find(".//country")
        ctry = _txt(ctry_el)
        iso = ""
        if ctry:
            iso = _LOOKUP.get(ctry.lower(), "") or (next(iter(countries_in(ctry)), ""))
        if not iso:
            iso = last_country_in_line(_txt(aff))     # fall back to text scan
        if not inst:
            inst = re.sub(r"\s+", " ", _txt(aff))[:200]
        affs[aid] = {"inst": inst, "iso": iso}
    if not affs:
        return None
    # contribs in order -> their aff refs
    authors = []
    for contrib in root.xpath(".//contrib[@contrib-type='author']") or root.xpath(".//contrib"):
        rids = [x.get("rid") for x in contrib.xpath(".//xref[@ref-type='aff']") if x.get("rid")]
        # some JATS put multiple ids in one rid attr
        rid_ids = []
        for r in rids:
            rid_ids.extend(r.split())
        c_isos = [affs[r]["iso"] for r in rid_ids if r in affs and affs[r]["iso"]]
        c_inst = [affs[r]["inst"] for r in rid_ids if r in affs and affs[r]["inst"]]
        authors.append({"iso": c_isos, "inst": c_inst})
    all_iso = sorted({i for a in affs.values() if a["iso"] for i in [a["iso"]]})
    insts = sorted({a["inst"] for a in affs.values() if a["inst"]})
    aligned = any(a["iso"] for a in authors)
    first_iso = authors[0]["iso"] if authors else []
    last_iso = authors[-1]["iso"] if authors else []
    return {"method": "jats", "author_aligned": aligned,
            "first": first_iso, "last": last_iso, "all": all_iso, "insts": insts}


# =========================================================================== #
# 2. GROBID TEI (author-aligned, for PDFs) — optional                         #
# =========================================================================== #
def from_tei(tei_text):
    try:
        tei_text = re.sub(r"<\?xml[^>]*\?>", "", tei_text)
        root = etree.fromstring(tei_text.encode("utf-8"), parser=etree.XMLParser(recover=True))
    except Exception:
        return None
    ns = {"t": "http://www.tei-c.org/ns/1.0"}
    authors = root.xpath("//t:sourceDesc//t:author", namespaces=ns)
    if not authors:
        return None
    seq = []
    for au in authors:
        isos = []
        for c in au.xpath(".//t:affiliation//t:country", namespaces=ns):
            code = c.get("key") or _LOOKUP.get(_txt(c).lower(), "") or last_country_in_line(_txt(c))
            if code: isos.append(code.upper())
        if not isos:  # country may be only in the raw affiliation text
            for aff in au.xpath(".//t:affiliation", namespaces=ns):
                code = last_country_in_line(_txt(aff))
                if code: isos.append(code)
        seq.append(sorted(set(isos)))
    insts = sorted({_txt(a) for a in root.xpath("//t:affiliation//t:orgName", namespaces=ns) if _txt(a)})
    all_iso = sorted({i for s in seq for i in s})
    aligned = any(seq)
    return {"method": "tei", "author_aligned": aligned,
            "first": seq[0] if seq else [], "last": seq[-1] if seq else [],
            "all": all_iso, "insts": insts[:20]}


# =========================================================================== #
# 3. front-matter regex on PDF/HTML text (paper-level country bag)            #
# =========================================================================== #
def from_text_frontmatter(sections):
    """Use the first ~60 lines (front matter) of the earliest big text block.
    To reduce name/country collisions (e.g. 'Chad', 'Georgia', 'Jordan' as given
    names), only scan lines that LOOK like affiliations (contain an org keyword or
    a comma-separated tail) and take the LAST country on each such line, which is
    where the country almost always sits."""
    text = ""
    for s in sections:
        if s.get("type") in ("title", "abstract"):
            continue
        text = s.get("text", ""); break
    if not text:
        text = " ".join(s.get("text", "") for s in sections[:2])
    head_lines = text.splitlines()[:60]

    ORG = re.compile(r"\b(university|universit[ée]|universidad|universidade|institut|"
                     r"institute|hospital|college|laboratory|laboratoire|department|"
                     r"d[ée]partement|faculty|facult[ée]|centre|center|ministry|school|"
                     r"academy|clinic|foundation|council|agency)\b", re.I)
    isos = set()
    insts = []
    for ln in head_lines:
        looks_aff = bool(ORG.search(ln)) or ln.count(",") >= 2
        if not looks_aff:
            continue
        c = last_country_in_line(ln)          # country sits at the end of aff lines
        if c:
            isos.add(c)
        if ORG.search(ln):
            insts.append(re.sub(r"\s+", " ", ln).strip()[:200])
    if not isos:
        return None
    return {"method": "regex", "author_aligned": False,
            "first": [], "last": [], "all": sorted(isos), "insts": insts[:10]}


# =========================================================================== #
# orchestration                                                                #
# =========================================================================== #
def extract_one(oa_id):
    # prefer the richest raw file we have
    xmlp = Path(RAW_DIR) / f"{oa_id}.xml"
    pdfp = Path(RAW_DIR) / f"{oa_id}.pdf"
    secp = Path(SEC_DIR) / f"{oa_id}.json"

    if xmlp.exists():
        r = from_jats(xmlp.read_text(encoding="utf-8", errors="ignore"))
        if r: return r
    if pdfp.exists() and GROBID_URL:
        try:
            import requests
            resp = requests.post(f"{GROBID_URL}/api/processHeaderDocument",
                                 files={"input": ("a.pdf", pdfp.read_bytes(), "application/pdf")},
                                 timeout=120)
            if resp.status_code == 200:
                r = from_tei(resp.text)
                if r: return r
        except Exception:
            pass
    if secp.exists():
        try:
            doc = json.loads(secp.read_text(encoding="utf-8"))
            r = from_text_frontmatter(doc.get("sections", []))
            if r: return r
        except Exception:
            pass
    return None


def oa_countries(rec):
    """Metadata countries for the side-by-side comparison."""
    for col in ("all_author_countries", "oa_countries"):
        if col in rec and S(rec[col]):
            return sorted({c for c in re.split(r"[;|]", S(rec[col])) if c})
    return []


if __name__ == "__main__":
    df = pd.read_csv(IN_PARQUET)
    print(f"[in] {len(df):,} rows", file=sys.stderr)
    rows = []
    n_ft = n_added = 0
    for _, rec in df.iterrows():
        oid = short_id(rec.get("oa_id"))
        r = extract_one(oid)
        oa_c = oa_countries(rec)
        if not r:
            rows.append({"oa_id": oid, "ft_source": "", "aff_method": "",
                         "author_aligned": False, "ft_first_author_countries": "",
                         "ft_last_author_countries": "", "ft_all_countries": "",
                         "ft_institutions": "", "oa_all_countries": ";".join(oa_c),
                         "added_countries": "", "agrees": None, "confidence": ""})
            continue
        n_ft += 1
        ft_all = r["all"]
        added = sorted(set(ft_all) - set(oa_c))
        agrees = bool(set(ft_all) & set(oa_c)) if oa_c else None
        conf = ("high" if r["author_aligned"] and r["method"] in ("jats", "tei")
                else "medium" if r["method"] in ("jats", "tei") else "low")
        if added and not oa_c:
            n_added += 1
        rows.append({"oa_id": oid, "ft_source": r["method"], "aff_method": r["method"],
                     "author_aligned": r["author_aligned"],
                     "ft_first_author_countries": ";".join(r["first"]),
                     "ft_last_author_countries": ";".join(r["last"]),
                     "ft_all_countries": ";".join(ft_all),
                     "ft_institutions": " | ".join(r["insts"][:10]),
                     "oa_all_countries": ";".join(oa_c),
                     "added_countries": ";".join(added),
                     "agrees": agrees, "confidence": conf})

    out = pd.DataFrame(rows)
    out.to_parquet(OUT_PARQUET, index=False)
    print(f"[write] {OUT_PARQUET}", file=sys.stderr)
    has_ft = out["ft_all_countries"].str.len() > 0
    no_oa = out["oa_all_countries"].str.len() == 0
    print(f"[coverage] full-text affiliations found: {has_ft.sum():,}/{len(out):,} "
          f"({has_ft.mean():.1%})", file=sys.stderr)
    print(f"[gain] studies with NO OpenAlex country now filled by full text: "
          f"{(has_ft & no_oa).sum():,}", file=sys.stderr)
    print(f"[quality] author-aligned (usable for leadership): "
          f"{out['author_aligned'].sum():,}", file=sys.stderr)
    print(f"[method]\n{out[out['aff_method']!='']['aff_method'].value_counts().to_string()}",
          file=sys.stderr)
    # disagreement audit (where both exist)
    both = out[(out["ft_all_countries"].str.len() > 0) & (out["oa_all_countries"].str.len() > 0)]
    if len(both):
        dis = (both["agrees"] == False).mean()
        print(f"[audit] of studies with BOTH sources, {dis:.1%} have zero country overlap "
              f"(inspect these)", file=sys.stderr)

In [ ]:
both

In [ ]:
out